# 5. Annotate prepared windows with local Qwen

Load the interim prepared-window manifests, run the local MLX Qwen3-VL model, and write two joinable outputs per video under `data/processed/<video-name>/`: `window_event_annotations.json` and `yolo_spatial_coordinates.json`. The coordinate file retains full frame-level YOLO `box_xyxy` values only for track IDs referenced by events.

This notebook must run natively on Apple Silicon with `mlx-vlm` installed; MLX/Metal is not available in the Airflow Docker container.

In [1]:
import json
from pathlib import Path

import mlx.core as mx
from videometa import LocalQwenEventAnnotator, PreparedWindowInput

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_ID = "mlx-community/Qwen3-VL-4B-Instruct-4bit"

mx.set_cache_limit(2 * 1024**3)
annotator = LocalQwenEventAnnotator(
    model_id=MODEL_ID,
    video_fps=2.0,
    max_tokens=600,
    prompt_token_budget=6000,
    feature_frames=8,
)

manifest_paths = sorted(INTERIM_DIR.glob("*/prepared_windows.json"))
if not manifest_paths:
    raise FileNotFoundError(f"No prepared window manifests found in {INTERIM_DIR}")

for manifest_path in manifest_paths:
    video_name = manifest_path.parent.name
    items = json.loads(manifest_path.read_text(encoding="utf-8"))
    output_dir = PROCESSED_DIR / video_name
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "window_event_annotations.json"

    results = []
    spatial_results = []
    spatial_output_path = output_dir / "yolo_spatial_coordinates.json"
    for item in items:
        prepared = PreparedWindowInput(
            start_seconds=item["start_seconds"],
            end_seconds=item["end_seconds"],
            object_features=tuple(item["object_features"]),
            frame_features=tuple(item["frame_features"]),
            image_messages=tuple(item.get("image_messages", [])),
            artifact_directory=item.get("artifact_directory"),
            annotated_video_path=item.get("annotated_video_path"),
        )
        try:
            events = annotator.annotate(prepared)
            result = {
                "start_seconds": prepared.start_seconds,
                "end_seconds": prepared.end_seconds,
                "annotated_video_path": prepared.annotated_video_path,
                "events": events,
            }
        except Exception as error:
            events = []
            result = {
                "start_seconds": prepared.start_seconds,
                "end_seconds": prepared.end_seconds,
                "annotated_video_path": prepared.annotated_video_path,
                "events": [],
                "error": str(error),
            }
        results.append(result)

        event_object_ids = {
            str(involved_object["id"])
            for event in events
            for involved_object in event.get("involved_objects", [])
            if involved_object.get("id") is not None
        }
        detections = []
        for frame in prepared.frame_features:
            for detection in frame.get("detections", []):
                if str(detection.get("track_id")) in event_object_ids:
                    detections.append(
                        {
                            "object_id": str(detection["track_id"]),
                            "frame_index": frame["frame_index"],
                            "timestamp_seconds": frame["timestamp_seconds"],
                            "label": detection["label"],
                            "confidence": detection["confidence"],
                            "box_xyxy": detection["box_xyxy"],
                            "spatial_position": detection["spatial_position"],
                        }
                    )
        spatial_results.append(
            {
                "start_seconds": prepared.start_seconds,
                "end_seconds": prepared.end_seconds,
                "event_object_ids": sorted(event_object_ids),
                "detections": detections,
            }
        )
        output_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
        spatial_output_path.write_text(json.dumps(spatial_results, indent=2), encoding="utf-8")

    print(f"Wrote {len(results)} window annotations to {output_path}")
    print(f"Wrote joinable YOLO coordinates to {spatial_output_path}")

/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 1697.22it/s]


Wrote 15 window annotations to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-05.13-15-00.13-20-00.bus.G340/window_event_annotations.json
Wrote joinable YOLO coordinates to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-05.13-15-00.13-20-00.bus.G340/yolo_spatial_coordinates.json
Wrote 15 window annotations to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-05.13-15-00.13-20-00.bus.G505/window_event_annotations.json
Wrote joinable YOLO coordinates to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-05.13-15-00.13-20-00.bus.G505/yolo_spatial_coordinates.json
Wrote 5 window annotations to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data/processed/2018-03-05.13-15-00.13-20-00.bus.G506/window_event_annotations.json
Wrote joinable YOLO coordinates to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/data